# OLTW

This notebook provides wrapper functions for calling the OLTW (Online Time Warping) algorithm using the PerformanceMatcher tool.  Running this algorithm requires installing Java and the PerformanceMatcher.jar file.  This notebook implements the `online_processing()` function, which will be imported and run in `02_RunExperiment.ipynb`.


Here is a summary of the OLTW approach:
- Offline processing: The piano reference audio is chopped to contain only the region of interest.
- Online processing: The query piano and chopped reference piano recordings are aligned using the OLTW algorithm, which outputs frame indices. These are then converted to seconds and adjusted to account for the reference offset.


## Offline Processing


In the offline processing stage, the piano reference audio is chopped to contain only the region of interest. This chopped audio file is stored in the scenario directory as `pref_chopped.wav`.


In [2]:
import numpy as np
import os
import os.path
import subprocess
import librosa as lb
from scipy.io.wavfile import write
import system_utils


In [ ]:
def offline_processing(scenario_dir, cache_dir, hop_length):
    '''
    Carries out offline processing for the OLTW system by chopping the piano reference audio.
    
    Inputs
    scenario_dir: The scenario directory to process
    cache_dir: The location of the cache directory (not used for OLTW, but kept for compatibility)
    hop_length: The hop length in samples used when computing features (not used for OLTW, but kept for compatibility)
    
    This function will create a chopped reference audio file in the scenario directory.
    '''
    # Verify scenario directory
    system_utils.verify_scenario_dir(scenario_dir)
    
    # Get piano reference boundaries
    pref_path = os.path.join(scenario_dir, "pref.wav")
    if not os.path.exists(pref_path):
        raise FileNotFoundError(f"pref.wav missing in {scenario_dir}")
    
    try:
        p_start_t, p_end_t = system_utils.get_piano_reference_boundaries(scenario_dir)
    except (AssertionError, FileNotFoundError, ValueError) as e:
        raise ValueError(f"Cannot find piano reference boundaries in scenario.info: {e}")
    
    # Load and write chopped file
    y_ref, sr_ref = lb.load(pref_path, sr=None)
    y_chopped = y_ref[int(p_start_t * sr_ref): int(p_end_t * sr_ref)]
    
    out_path = os.path.join(scenario_dir, "pref_chopped.wav")
    write(out_path, sr_ref, (y_chopped * 32767).astype("int16"))  # 16-bit PCM
    
    return


In [4]:
def verify_cache_dir(indir):
    '''
    Verifies that the specified cache directory exists.
    For OLTW, we don't actually use a cache, but this function is kept for compatibility.
    
    Inputs
    indir: The cache directory to verify
    '''
    # OLTW doesn't use a cache directory, so we just check if it exists
    if not os.path.exists(indir):
        os.makedirs(indir)
    return


## Online Processing


In the online processing stage, we:
1. verify that the chopped reference audio exists (created in offline processing),
2. run the OLTW algorithm to align the query piano and chopped reference piano recordings,
3. parse the alignment output and convert from frame indices to seconds,
4. adjust the reference times to account for the offset in the full reference audio.


### Software Installation


Using the OLTW algorithm requires:
- Java runtime environment (JRE) installed and accessible from command line
- The PerformanceMatcher.jar file, which should be located in the `match/` directory


Below, we will assume that `java` can be called from command line, and that PerformanceMatcher.jar is available in the expected location.


### Wrapper Implementation


In [ ]:
def verify_oltw_installation(jar_path):
    '''Verifies that all tools needed to run OLTW are present
    
    Inputs
    jar_path: Path to the PerformanceMatcher.jar file
    '''
    from shutil import which
    import os
    
    # Prefer conda Java if available (usually has the right version)
    java_cmd = None
    conda_prefix = os.environ.get('CONDA_PREFIX')
    if conda_prefix:
        conda_java = os.path.join(conda_prefix, 'bin', 'java')
        if os.path.exists(conda_java):
            java_cmd = conda_java
    
    # Fall back to system Java
    if java_cmd is None:
        java_cmd = which('java')
    
    assert java_cmd is not None, 'Java is not installed or not in PATH. Please install Java runtime environment.'
    assert os.path.exists(jar_path), f'PerformanceMatcher.jar not found at {jar_path}. Please ensure the JAR file is in the correct location.'
    
    # Check Java version - PerformanceMatcher.jar requires Java 21 (class file version 65.0)
    try:
        result = subprocess.run([java_cmd, '-version'], 
                               stdout=subprocess.PIPE, 
                               stderr=subprocess.STDOUT, 
                               timeout=5,
                               text=True)
        version_output = result.stdout + result.stderr if result.stderr else result.stdout
        
        # Check if Java 21 or higher
        import re
        version_match = re.search(r'version "(\d+)\.', version_output)
        if version_match:
            java_version = int(version_match.group(1))
            if java_version < 21:
                raise RuntimeError(
                    f'Java version {java_version} is too old. PerformanceMatcher.jar requires Java 21 or higher. '
                    f'Current Java: {java_cmd}\nVersion output: {version_output}'
                )
        else:
            print(f'Warning: Could not parse Java version. Output: {version_output}')
    except subprocess.TimeoutExpired:
        pass  # If it times out, that's okay
    except Exception as e:
        if 'too old' in str(e):
            raise
        print(f'Warning: Could not verify Java version: {e}')
    
    # Try to run java -jar to verify the JAR is valid
    try:
        result = subprocess.run([java_cmd, '-jar', jar_path], 
                               stdout=subprocess.PIPE, 
                               stderr=subprocess.PIPE, 
                               timeout=5)
    except subprocess.TimeoutExpired:
        pass  # If it runs, that's fine
    except Exception as e:
        raise RuntimeError(f'Could not run PerformanceMatcher.jar: {e}')
    
    return java_cmd


In [6]:
def parse_oltw_alignment(infile):
    '''
    Parses the OLTW alignment text output file.
    
    Inputs
    infile: filepath to the OLTW alignment text output file
    
    Returns a list of tuples (query_frame, ref_frame) in frame indices.
    OLTW outputs query frames as 1-based indices and reference frames as 0-based indices
    relative to the chopped reference audio.
    '''
    alignment_data = []
    with open(infile, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]
    
    for line in lines:
        if line.startswith("ALIGNMENT"):
            parts = line.split(" ")
            if len(parts) >= 3:
                # Parse query index (remove trailing comma if present)
                query_idx = int(parts[1].rstrip(','))
                # Parse reference index
                ref_idx = int(parts[2])
                alignment_data.append((query_idx, ref_idx))
    
    return alignment_data


In [ ]:
def online_processing(scenario_dir, out_dir, cache_dir, hop_length, jar_path=None):
    '''
    Carries out online processing using the OLTW algorithm.
    
    Inputs
    scenario_dir: The scenario directory to process
    out_dir: The directory to put results, intermediate files, and logging info
    cache_dir: The cache directory (not used for OLTW, but kept for compatibility)
    hop_length: The hop length in samples (default 512 for OLTW)
    jar_path: Path to PerformanceMatcher.jar. If None, will look in match/PerformanceMatcher.jar

    This function will compute and save the predicted alignment in the output directory in a file hyp.npy
    The alignment is saved in seconds, with reference times relative to the full reference audio.
    '''
    
    # Set default jar path if not provided
    if jar_path is None:
        # Try to find the jar file relative to current directory
        possible_paths = [
            'match/PerformanceMatcher.jar',
            '../match/PerformanceMatcher.jar',
            os.path.join(os.path.dirname(os.getcwd()), 'match', 'PerformanceMatcher.jar')
        ]
        jar_path = None
        for path in possible_paths:
            if os.path.exists(path):
                jar_path = path
                break
        if jar_path is None:
            raise FileNotFoundError('Could not find PerformanceMatcher.jar. Please specify jar_path.')
    
    # Verify installation
    verify_oltw_installation(jar_path)
    
    # Verify & setup
    system_utils.verify_scenario_dir(scenario_dir)
    verify_cache_dir(cache_dir)
    assert not os.path.exists(out_dir), f'Output directory {out_dir} already exists.'
    os.makedirs(out_dir)
    
    # Check that chopped reference exists (should have been created in offline processing)
    pref_chopped_path = os.path.join(scenario_dir, 'pref_chopped.wav')
    if not os.path.exists(pref_chopped_path):
        raise FileNotFoundError(f'pref_chopped.wav not found in {scenario_dir}. Please run offline_processing first.')
    
    # Get piano reference boundaries for offset calculation
    pref_start_sec, pref_end_sec = system_utils.get_piano_reference_boundaries(scenario_dir)
    
    # Set up file paths
    query_path = os.path.join(scenario_dir, 'p.wav')
    ref_path = pref_chopped_path
    alignment_output_path = os.path.join(out_dir, 'oltw_alignment.txt')
    
    # Run OLTW algorithm
    cmd = [
        'java', '-jar', jar_path,
        '-b', '-q', '-G', '-D', '--use-chroma-map',
        query_path, ref_path
    ]
    
    with open(alignment_output_path, 'w') as f:
        result = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, check=False)
    
    if result.returncode != 0:
        print(f'Warning: OLTW returned non-zero exit code. Check {alignment_output_path} for details.')
    
    # Parse alignment output
    alignment_data = parse_oltw_alignment(alignment_output_path)
    
    if len(alignment_data) == 0:
        raise RuntimeError(f'No alignment data found in {alignment_output_path}')
    
    # Convert frame indices to seconds and adjust reference offset
    hop_sec = hop_length / 22050.0  # Assuming 22050 Hz sample rate
    
    # Convert to seconds: query frames are 1-based, reference frames are 0-based relative to chopped audio
    # We subtract 1 from query frames to convert to 0-based, then multiply by hop_sec
    # For reference, we multiply by hop_sec and add pref_start_sec to get time in full reference
    alignment_seconds = []
    for query_frame, ref_frame in alignment_data:
        query_sec = (query_frame - 1) * hop_sec  # Convert 1-based to 0-based, then to seconds
        ref_sec = ref_frame * hop_sec + pref_start_sec  # Convert to seconds and add offset
        alignment_seconds.append((query_sec, ref_sec))
    
    # Convert to numpy array and transpose to match expected format (2 x N)
    alignment_array = np.array(alignment_seconds).T
    
    # Save alignment
    np.save(os.path.join(out_dir, 'hyp.npy'), alignment_array)
    
    # Clean up intermediate files
    if os.path.exists(alignment_output_path):
        os.remove(alignment_output_path)
    
    return


In [8]:
def verify_hyp_dir(indir):
    '''
    Verifies that the specified scenario hypothesis directory has the required files.
    
    Inputs
    indir: The hypothesis directory to verify
    '''
    assert os.path.exists(os.path.join(indir, 'hyp.npy')), f'{indir} is missing hyp.npy, please re run the online processing'
